In [ ]:
from pyspark.sql import functions as F

SRC = "Files/bronze/air_quality/*.json"

raw = (spark.read.option("multiLine", True).json(SRC)
       .select("*", F.col("_metadata.file_path").alias("file_path")))

flat = (raw
  .withColumn("file_name", F.element_at(F.split("file_path", "/"), -1))
  .withColumn("city", F.regexp_extract("file_name", r"^(.*)_(\d{12})\.json$", 1))
  .withColumn("file_ts", F.to_timestamp(
      F.regexp_extract("file_name", r"_(\d{12})\.json$", 1), "yyyyMMddHHmm"))
  .withColumn("h", F.explode(F.arrays_zip(
      "hourly.time", "hourly.pm10", "hourly.pm2_5",
      "hourly.nitrogen_dioxide", "hourly.ozone", "hourly.european_aqi")))
  .select("city", "file_name", "file_ts",
          F.col("latitude").alias("lat_grille"),
          F.col("longitude").alias("lon_grille"),
          F.to_timestamp("h.time").alias("time"),
          F.col("h.pm10").alias("pm10"),
          F.col("h.pm2_5").alias("pm2_5"),
          F.col("h.nitrogen_dioxide").alias("nitrogen_dioxide"),
          F.col("h.ozone").alias("ozone"),
          F.col("h.european_aqi").alias("european_aqi")))

silver

In [ ]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

SILVER = "silver_air_quality"
LOG = "silver_air_quality_log"
VARS = ["pm10", "pm2_5", "nitrogen_dioxide", "ozone", "european_aqi"]

# 1) Point de reprise
if spark.catalog.tableExists(LOG):
    last_ts = spark.table(LOG).agg(F.max("max_file_ts")).first()[0]
else:
    last_ts = None
print("Point de reprise :", last_ts)

new = flat if last_ts is None else flat.filter(F.col("file_ts") > F.lit(last_ts))
n_files = new.select("file_name").distinct().count()

if n_files == 0:
    print("Rien de nouveau.")
else:
    # 2) Qualité : lignes entièrement vides = hors horizon (comptées à part)
    all_null = F.col(VARS[0]).isNull()
    for c in VARS[1:]:
        all_null = all_null & F.col(c).isNull()

    hors_horizon = new.filter(all_null).count()
    valid = new.filter(~all_null)

    # 3) Déduplication (city, time) : le fichier le plus récent gagne
    w = Window.partitionBy("city", "time").orderBy(F.col("file_ts").desc())
    dedup = (valid.withColumn("rn", F.row_number().over(w))
                  .filter("rn = 1").drop("rn"))

    # 4) Complétude : les 7 villes sont-elles présentes ?
    villes = [r.city for r in new.select("city").distinct().collect()]
    complet = len(villes) == 7

    # 5) MERGE conditionnel (mise à jour seulement si le fichier est plus récent)
    if spark.catalog.tableExists(SILVER):
        (DeltaTable.forName(spark, SILVER).alias("t")
            .merge(dedup.alias("s"), "t.city = s.city AND t.time = s.time")
            .whenMatchedUpdateAll(condition="s.file_ts > t.file_ts")
            .whenNotMatchedInsertAll()
            .execute())
    else:
        dedup.write.format("delta").saveAsTable(SILVER)

    # 6) Journal qualité
    max_ts = new.agg(F.max("file_ts")).first()[0]
    log = spark.createDataFrame(
        [(max_ts, n_files, new.count(), hors_horizon, len(villes), complet)],
        "max_file_ts timestamp, fichiers int, lignes_lues long, hors_horizon long, villes int, complet boolean"
    ).withColumn("run_ts", F.current_timestamp())
    log.write.format("delta").mode("append").saveAsTable(LOG)

    print(f"OK : {n_files} fichiers, {hors_horizon} hors horizon, complet={complet}")

print("silver rows :", spark.table(SILVER).count())
spark.table(LOG).show(truncate=False)

Gold

In [ ]:
from pyspark.sql import functions as F

GOLD = "gold_air_quality_daily"

gold = (spark.table("silver_air_quality")
    .withColumn("date", F.to_date("time"))
    .groupBy("city", "date")
    .agg(
        F.round(F.avg("pm10"), 1).alias("pm10_moy"),
        F.round(F.avg("pm2_5"), 1).alias("pm2_5_moy"),
        F.round(F.avg("nitrogen_dioxide"), 1).alias("no2_moy"),
        F.round(F.avg("ozone"), 1).alias("ozone_moy"),
        F.round(F.avg("european_aqi"), 1).alias("aqi_moy"),
        F.max("european_aqi").alias("aqi_max"),
        F.count("*").alias("nb_heures"),
    ))

(gold.write.format("delta")
     .mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(GOLD))

print("gold rows :", spark.table(GOLD).count())
spark.table(GOLD).orderBy("city", "date").show(12, truncate=False)
spark.table(GOLD).groupBy("nb_heures").count().orderBy("nb_heures").show()

In [ ]:
#spark.table("gold_weather_daily").printSchema()